In [7]:
import numpy as np
import pandas as pd

df=pd.read_csv('tmdb_movies_dataset.csv')

In [8]:
# df.info()
# df.head()
df.describe()
# df.isna().sum()

,id,popularity,vote_average,vote_count
count,1.802200e+04,18022.000000,18022.000000,18022.000000
mean,5.845825e+05,3.308476,5.345313,1145.538619
std,5.776051e+05,14.393934,2.800091,2595.960450
min,2.000000e+00,0.000000,0.000000,0.000000
25%,1.878200e+04,0.818150,5.204250,2.000000
50%,4.205665e+05,1.934200,6.350500,359.000000
75%,1.167855e+06,3.118175,7.063000,1006.000000
max,1.516417e+06,1075.835800,10.000000,37686.000000


In [9]:
# remove 'original_title' due to multilinguality
df=df.drop(columns=['original_title'])

df=df.dropna(subset=['release_date'])

# fill 'overview' with no description for later Naive Bayes classification
df['overview']=df['overview'].fillna("no description available.")

# convert 'release_date' to datetime for conversion
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# Create separate columns for year, month, and day
df['release_year']=df['release_date'].dt.year
df['release_decade']=(df['release_year'] // 10) * 10
df['release_month']=df['release_date'].dt.month
df['release_day']=df['release_date'].dt.day

In [10]:
import ast

df['genre_ids'] = df['genre_ids'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Explode genre_ids so each genre becomes its own row
df=df.explode('genre_ids').reset_index(drop=True)

# one-hot encode
df=pd.get_dummies(df, columns=['genre_ids'], prefix='genre')

In [11]:
# popularity - severly max outlier, right-skewed
# vote_average - min & max outlier, left-skewed
# vote_count - severly imbalanced, right-skewed

# release_date - severely min outlier, severely left-skewed
# release_year - severely min outlier, severely left-skewed
# release_decade - min outlier, left-skewed

In [12]:
import seaborn as sns
import matplotlib.pyplot as plt

cols=['popularity', 'vote_average', 'vote_count']
categorical_cols=['genre_ids', 'original_language']
numerical_cols=['popularity', 'vote_average', 'vote_count']

# for col in cols:
    # sns.histplot(df[col], kde=True)
    # plt.title(f'Histplot - {col}')
    # plt.show()

    # sns.boxplot(df[col])
    # plt.title(f'Boxplot - {col}')
    # plt.show()

# sns.countplot(x='genre_ids', data=df_exploded, order=df_exploded['genre_ids'].value_counts().index)
# plt.title('Genre Frequency')
# plt.xlabel('Genre ID')
# plt.ylabel('Number of Movies')
# plt.xticks(rotation=45)
# plt.show()

In [13]:
for col in cols:
    Q1=df[col].quantile(0.25)
    Q3=df[col].quantile(0.75)
    IQR=Q3-Q1

    lower_bound=Q1-1.5*IQR
    upper_bound=Q3+1.5*IQR

    min_value=df[col].min()
    max_value=df[col].max()

    if min_value < lower_bound or max_value > upper_bound:
        df[f'{col}_capped']=df[col].clip(lower=lower_bound, upper=upper_bound)

In [14]:
from scipy.stats import skew

cols_to_log=['popularity', 'vote_count']

for col in cols_to_log:
    skewness=skew(df[col])
    print(f"Skewness Before[{col}]: {skewness}")
    df[f'{col}_log'] = np.log1p(df[col])
    skewness=skew(df[f'{col}_log'])
    print(f"Skewness Now[{col}]: {skewness}")

Skewness Before[popularity]: 37.72590907922011
Skewness Now[popularity]: 0.9957278777390214
Skewness Before[vote_count]: 4.672318732019679
Skewness Now[vote_count]: -0.8019365438769556


In [15]:
df=pd.get_dummies(df, columns=['original_language'], drop_first=True)

In [16]:
df=df.drop(columns=['vote_average', 'popularity_capped', 'overview', 'id', 'title'], errors='ignore')
# df.head()

In [17]:
# corr=df.corr(numeric_only=True)
# plt.figure(figsize=(20, 14))
# sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
# plt.title('Feature Correlation Heatmap')
# plt.show()

In [18]:
import pandas as pd
import ast
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, hamming_loss, f1_score
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# 1️⃣ Load dataset
df = pd.read_csv('tmdb_movies_dataset.csv')

# 2️⃣ Process genre_ids (list column ➔ one-hot encoding)
df['genre_ids'] = df['genre_ids'].apply(lambda x: ast.literal_eval(str(x)))

mlb = MultiLabelBinarizer()
genre_dummies = pd.DataFrame(
    mlb.fit_transform(df['genre_ids']),
    columns=[f'genre_{g}' for g in mlb.classes_]
)

df = pd.concat([df, genre_dummies], axis=1)

# Drop movies without any genre
genre_columns = genre_dummies.columns.tolist()
df = df[df[genre_columns].sum(axis=1) > 0]

# 3️⃣ Create primary_genre (multi-label ➔ single-label)
df['primary_genre'] = df[genre_columns].idxmax(axis=1)

# ✅ Remove rare genres (classes with < 2 samples)
genre_counts = df['primary_genre'].value_counts()
rare_genres = genre_counts[genre_counts < 2].index

df = df[~df['primary_genre'].isin(rare_genres)]
print("Removed rare genres:", list(rare_genres))
print("Class distribution after cleaning:\n", df['primary_genre'].value_counts())

# 4️⃣ Define X (drop problematic columns)
X = df.drop(columns=['primary_genre'], errors='ignore')

datetime_cols = X.select_dtypes(include=['datetime64']).columns
X = X.drop(columns=datetime_cols, errors='ignore')

non_numeric_cols = X.select_dtypes(include=['object']).columns
X = X.drop(columns=non_numeric_cols, errors='ignore')

print("Remaining columns in X:", X.columns)

# 5️⃣ Define y
y = df['primary_genre']

# 6️⃣ Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

# 7️⃣ SMOTE oversampling
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("\nAfter SMOTE:\n", y_train_resampled.value_counts())

# 8️⃣ CatBoost + Grid Search
model = CatBoostClassifier(
    verbose=False,
    random_state=42,
    auto_class_weights='Balanced'
)

param_grid = {
    'depth': [4, 6, 8],
    'iterations': [300, 500, 1000],
    'learning_rate': [0.01, 0.03, 0.1],
    'l2_leaf_reg': [1, 3, 5]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train_resampled, y_train_resampled)

# 9️⃣ Evaluation
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

print("Best Parameters:", grid_search.best_params_)
print("\nClassification Report:\n", classification_report(y_test, predictions, zero_division=0))
print("Hamming Loss:", hamming_loss(y_test, predictions))
print("F1 Score (micro):", f1_score(y_test, predictions, average='micro'))
print("F1 Score (macro):", f1_score(y_test, predictions, average='macro'))

Removed rare genres: ['genre_10770']
Class distribution after cleaning:
 primary_genre
genre_18       6694
genre_12       2170
genre_35       1902
genre_27       1724
genre_28       1116
genre_14        999
genre_99        934
genre_16        645
genre_53        499
genre_10749     136
genre_10402     108
genre_36         61
genre_878        52
genre_37         45
genre_80         39
genre_9648       26
genre_10751      20
genre_10752       7
Name: count, dtype: int64
Remaining columns in X: Index(['id', 'popularity', 'vote_average', 'vote_count', 'genre_12',
       'genre_14', 'genre_16', 'genre_18', 'genre_27', 'genre_28', 'genre_35',
       'genre_36', 'genre_37', 'genre_53', 'genre_80', 'genre_99', 'genre_878',
       'genre_9648', 'genre_10402', 'genre_10749', 'genre_10751',
       'genre_10752', 'genre_10770'],
      dtype='object')
Before SMOTE:
 primary_genre
genre_18       5355
genre_12       1736
genre_35       1521
genre_27       1379
genre_28        893
genre_14        799


C:\Users\dreyyan\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\resource_tracker.py:120: UserWarning: resource_tracker: process died unexpectedly, relaunching.  Some folders/sempahores might leak.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
print("Accuracy:", accuracy_score(y_test, predictions))